# Olist: tracking thí nghiệm và model registry với MLflow

Model dự đoán đơn bị **review xấu** (`review_score <= 2`), chấm **lúc đơn vừa giao xong**: được dùng
các thông tin giao hàng (số ngày giao thật, số ngày trễ so với hẹn), không được dùng bất kỳ cột nào
sinh ra từ review. Chia theo thời gian mua: train trước 2018-01-01, validation 2018-01-01 tới trước
2018-04-01, test 2018-04-01 tới hết 2018-08-31.

Bạn sẽ: ghi mỗi lần train thành một run MLflow (params, metric, model kèm signature), chọn run tốt
nhất trên **validation**, đăng ký vào Model Registry với alias `champion`, rồi nạp lại model theo
alias để chấm test, đúng cách một service production nạp model.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm.
- Cell gắn tag `explore` (dòng đầu `# explore`) đọc dữ liệu thật và gọi hàm; test bỏ qua chúng.
- MLflow ghi vào một thư mục tạm (SQLite), không ghi gì vào repo.
- Chấm: `uv run pytest tracks/data-science/modules/14-mlops-for-ds/exercises/intermediate/01-olist-tracking-registry -q`

In [ ]:
import logging
import os
import warnings

warnings.filterwarnings("ignore", message="IProgress not found")       # thanh tiến trình tqdm trong Jupyter
os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1")
os.environ.setdefault("MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR", "false")   # bớt dòng tiến trình khi nạp model

from pathlib import Path

import mlflow
import numpy as np
import pandas as pd
from mlflow import MlflowClient
from mlflow.models import infer_signature
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

logging.getLogger("mlflow.utils.environment").setLevel(logging.ERROR)  # môi trường uv không có pip: cảnh báo vô hại

In [ ]:
# Cell viết sẵn: bảng một dòng mỗi đơn giao xong có review, feature có sẵn lúc đơn vừa giao.
DATES = ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
         "order_delivered_customer_date", "order_estimated_delivery_date"]
NUMERIC = ["promised_days", "delivery_days", "delay_days", "approve_hours", "n_items", "n_sellers",
           "price", "freight", "freight_ratio", "payment_installments"]
CATEGORICAL = ["customer_state", "payment_type"]
FEATURES = NUMERIC + CATEGORICAL
TARGET = "bad_review"


def load_olist_table(raw) -> pd.DataFrame:
    raw = Path(raw)
    orders = pd.read_csv(raw / "olist_orders_dataset.csv", parse_dates=DATES)
    reviews = pd.read_csv(raw / "olist_order_reviews_dataset.csv", parse_dates=["review_answer_timestamp"])
    customers = pd.read_csv(raw / "olist_customers_dataset.csv")
    items = pd.read_csv(raw / "olist_order_items_dataset.csv")
    payments = pd.read_csv(raw / "olist_order_payments_dataset.csv")

    delivered = orders[(orders.order_status == "delivered") & orders.order_delivered_customer_date.notna()]
    latest = reviews.sort_values("review_answer_timestamp", kind="stable").drop_duplicates("order_id", keep="last")
    df = delivered.merge(latest[["order_id", "review_score"]], on="order_id").merge(
        customers[["customer_id", "customer_state"]], on="customer_id")
    df[TARGET] = (df.review_score <= 2).astype(int)

    day = pd.Timedelta(days=1)
    df["promised_days"] = (df.order_estimated_delivery_date - df.order_purchase_timestamp) / day
    df["delivery_days"] = (df.order_delivered_customer_date - df.order_purchase_timestamp) / day
    df["delay_days"] = (df.order_delivered_customer_date - df.order_estimated_delivery_date) / day
    df["approve_hours"] = (df.order_approved_at - df.order_purchase_timestamp) / pd.Timedelta(hours=1)
    per_order = items.groupby("order_id").agg(
        n_items=("order_item_id", "size"), n_sellers=("seller_id", "nunique"),
        price=("price", "sum"), freight=("freight_value", "sum"))
    df = df.merge(per_order, on="order_id", how="left")
    df["freight_ratio"] = df.freight / (df.price + df.freight)
    main = payments.sort_values("payment_value", ascending=False, kind="stable").drop_duplicates("order_id")
    df = df.merge(main[["order_id", "payment_type", "payment_installments"]], on="order_id", how="left")
    df[CATEGORICAL] = df[CATEGORICAL].fillna("missing")
    df[NUMERIC] = df[NUMERIC].astype("float64")   # hợp đồng đầu vào: mọi cột số là float64
    return df.sort_values("order_purchase_timestamp", kind="stable").reset_index(drop=True)


def time_split(df: pd.DataFrame) -> dict[str, pd.DataFrame]:
    t = df.order_purchase_timestamp
    return {"train": df[t < "2018-01-01"],
            "val": df[(t >= "2018-01-01") & (t < "2018-04-01")],
            "test": df[(t >= "2018-04-01") & (t < "2018-09-01")]}


def build_pipeline(C: float = 1.0) -> Pipeline:
    pre = ColumnTransformer([
        ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), NUMERIC),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL)])
    return make_pipeline(pre, LogisticRegression(C=C, max_iter=2000))

In [ ]:
# explore: đọc Olist thật (tải bằng: uv run python data/download_olist.py)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
table = load_olist_table(ROOT / "data" / "raw")
splits = time_split(table)
train, val, test = splits["train"], splits["val"], splits["test"]
print(table.shape, {k: len(v) for k, v in splits.items()})
print({k: round(v[TARGET].mean(), 4) for k, v in splits.items()})

In [ ]:
# explore: một backend SQLite trong thư mục tạm, artifact cũng nằm ở đó (không tạo mlruns/ trong repo)
import tempfile

STORE = Path(tempfile.mkdtemp(prefix="olist-mlflow-"))
TRACKING_URI = f"sqlite:///{STORE / 'mlflow.db'}"
client = MlflowClient(TRACKING_URI)
EXPERIMENT_ID = client.create_experiment("olist-bad-review", artifact_location=(STORE / "artifacts").as_uri())
print(TRACKING_URI)

## 1. Mỗi lần train là một run

`train_and_log` fit `build_pipeline(C)` trên `train`, chấm `val`, và ghi vào một run của
`experiment_id`:

- param `C` và `n_train` (số dòng train),
- metric `val_auc`,
- model scikit-learn ở `name="model"`, kèm **signature** suy ra từ `train[FEATURES]` và xác suất dự
  đoán, `input_example` vài dòng, và `pyfunc_predict_fn="predict_proba"` để khi nạp qua
  `mlflow.pyfunc` thì `predict` trả xác suất chứ không phải nhãn 0/1.

Trả `run_id`. Nên ghi thêm tag `model_uri` (giá trị `info.model_uri` mà `log_model` trả về): ở
MLflow 3, model là một *LoggedModel* riêng có URI `models:/m-...`, và đăng ký từ URI đó gọn hơn
đường dẫn cũ `runs:/<run_id>/model`.

In [ ]:
def train_and_log(train: pd.DataFrame, val: pd.DataFrame, C: float, tracking_uri: str,
                  experiment_id: str) -> str:
    """Fit build_pipeline(C) trên train, ghi param/metric/model vào một run MLflow. Trả run_id."""
    mlflow.set_tracking_uri(tracking_uri)
    with mlflow.start_run(experiment_id=experiment_id) as run:
        pipe = build_pipeline(C).fit(train[FEATURES], train[TARGET])
        val_auc = roc_auc_score(val[TARGET], pipe.predict_proba(val[FEATURES])[:, 1])
        mlflow.log_params({"C": C, "n_train": len(train)})
        mlflow.log_metric("val_auc", float(val_auc))
        # Suy signature trên cả tập train (có NaN thật) để cột hay thiếu được đánh dấu optional.
        # Cột số ép float: MLflow không tự đổi int64 sang float64 khi kiểm schema.
        X = train[FEATURES].astype({c: "float64" for c in NUMERIC})
        signature = infer_signature(X, pipe.predict_proba(X))
        info = mlflow.sklearn.log_model(
            pipe, name="model", signature=signature, input_example=X.head(3),
            pyfunc_predict_fn="predict_proba", pip_requirements=["scikit-learn", "pandas"],
            skops_trusted_types=["numpy.dtype"],   # định dạng skops (mặc định, an toàn hơn pickle) cần khai báo kiểu
        )
        mlflow.set_tag("model_uri", info.model_uri)   # MLflow 3: model là LoggedModel riêng, URI dạng models:/m-...
    return run.info.run_id

In [ ]:
# explore: vài giá trị C, mỗi giá trị một run
run_ids = {C: train_and_log(train, val, C, TRACKING_URI, EXPERIMENT_ID) for C in [0.01, 0.1, 1.0, 10.0]}
runs = client.search_runs([EXPERIMENT_ID], order_by=["metrics.val_auc DESC"])
pd.DataFrame([{"run_id": r.info.run_id[:8], "C": r.data.params["C"], "val_auc": round(r.data.metrics["val_auc"], 4)}
              for r in runs])

## 2. Model Registry: đăng ký và gắn alias

`register_best` tìm run `FINISHED` có `val_auc` cao nhất trong experiment, đăng ký model của run đó
(tag `model_uri` nếu có, nếu không thì `runs:/<run_id>/model`) dưới tên `model_name`, gắn alias `champion` cho version mới, và trả số
version (`int`). Gọi lại sau khi có run tốt hơn thì tạo version mới và alias chuyển sang version đó;
version cũ vẫn còn để rollback.

`load_champion` nạp `models:/<model_name>@champion` bằng `mlflow.pyfunc.load_model`. Service nạp
model theo **alias**, không theo số version: đổi model là đổi alias, không phải sửa code service.

In [ ]:
def register_best(tracking_uri: str, experiment_id: str, model_name: str, metric: str = "val_auc") -> int:
    """Đăng ký model của run tốt nhất theo metric, gắn alias champion, trả số version."""
    client = MlflowClient(tracking_uri)
    runs = client.search_runs(
        [experiment_id],
        filter_string=f"attributes.status = 'FINISHED' and metrics.{metric} > -1e308",
        order_by=[f"metrics.{metric} DESC"], max_results=1)
    if not runs:
        raise ValueError(f"Không có run nào có {metric}")
    best = runs[0]
    mlflow.set_tracking_uri(tracking_uri)       # register_model dùng tracking/registry URI toàn cục
    source = best.data.tags.get("model_uri", f"runs:/{best.info.run_id}/model")
    version = mlflow.register_model(source, model_name)
    client.set_registered_model_alias(model_name, "champion", version.version)
    client.set_model_version_tag(model_name, version.version, metric, str(best.data.metrics[metric]))
    return int(version.version)


def load_champion(tracking_uri: str, model_name: str):
    """Nạp model đang mang alias champion dưới dạng pyfunc."""
    mlflow.set_tracking_uri(tracking_uri)
    return mlflow.pyfunc.load_model(f"models:/{model_name}@champion")

In [ ]:
# explore: đăng ký, nạp theo alias, chấm test (lần đầu và duy nhất nhìn test)
version = register_best(TRACKING_URI, EXPERIMENT_ID, "olist-bad-review")
champion = load_champion(TRACKING_URI, "olist-bad-review")
proba = np.asarray(champion.predict(test[FEATURES]))[:, 1]
print("version", version, "| test AUC:", round(roc_auc_score(test[TARGET], proba), 4))
print(champion.metadata.get_input_schema())

In [ ]:
# explore: signature được kiểm khi predict qua pyfunc. Thiếu cột thì báo lỗi ngay, không đoán bừa.
try:
    champion.predict(test[FEATURES].drop(columns=["delay_days"]).head())
except Exception as err:
    print(type(err).__name__, "...", str(err).strip().splitlines()[-1][:200])

## 3. Câu hỏi để tự trả lời

1. Bốn giá trị C cho `val_auc` chênh nhau bao nhiêu? Chênh lệch đó có đáng để chọn C "tốt nhất"
   không, hay chỉ là nhiễu?
2. AUC trên test thấp hơn hẳn validation. Đó là model tệ đi, hay tập test khác validation? (Gợi ý:
   so tỉ lệ đơn giao trễ ở hai giai đoạn; README module, mục monitoring.)
3. Trước khi đưa champion ra production, bạn có fit lại trên train + validation không? Nếu có thì
   run nào trong MLflow chứa model đó, và metric nào đi kèm nó?